In [1]:
import os
import plotly.express as px
from pandas import DataFrame
from torch import load, stack, cat, linalg
import random
from collections import defaultdict

from utils import centroid

In [2]:
datasets = {
    'core_clinical': {
        'folder': 'core_clinical_short_gemma3_27b_21/',
        'files': 'core_clinical_short'
    },
    'books': {
        'folder': 'books_gemma3_27b_21/',
        'files': 'books'
    },
    'redsm5': {
        'folder': 'redsm5_gemma3_27b_21/',
        'files': 'redsm5'
    }
}

In [3]:
# Build symptom anchors from the up-to-date short activations rather than the
# stale precomputed '61.gemma3_27b_anchors'. For each core clinical text we take
# its residual-stream embeddings (dropping the leading BOS position) and pool
# them by clinician annotation, so anchor[symptom] is every position of that
# symptom stacked (n, d_model) — the same structure the old file held.
cc = datasets['core_clinical']
anchor_folder, anchor_dir = cc['folder'], cc['files']

anchor_rs = {}
for text in os.listdir(anchor_dir):
    symptom = text.split(",")[1]
    rs = load(anchor_folder + text + '_tensor')[0][1:]  # drop BOS
    anchor_rs.setdefault(symptom, []).append(rs)

anchor = {symptom: cat(embs) for symptom, embs in anchor_rs.items()}
print({k: tuple(v.shape) for k, v in anchor.items()})

{'mood': (1110, 5376), 'somatic': (812, 5376), 'suicidality': (243, 5376)}


In [4]:
# Calculating symptom vectors from origin
# float64 BEFORE averaging: the stored activations are bfloat16, and a bf16
# mean then cast to double loses ~0.1% on the centroid norm.
cent = {k: centroid(v.double()) for k, v in anchor.items()}
print(cent)

{'mood': tensor([ -3.3316,  -3.1023, -11.6711,  ...,  -4.2627,   0.3722,  -9.9976],
       dtype=torch.float64), 'somatic': tensor([ -3.7057,  -6.8720, -14.3042,  ...,  -2.3473,   4.5505,  -2.2824],
       dtype=torch.float64), 'suicidality': tensor([  0.6307,  -1.0334,  -8.6470,  ...,  -2.0332,  -0.1224, -14.8360],
       dtype=torch.float64)}


In [5]:
# Gram matrix
symptom_stack = stack([cent["mood"], cent["somatic"], cent["suicidality"]])

gram = symptom_stack @ symptom_stack.T
print(gram)
print(f"cond(G) = {linalg.cond(gram).item():.1f}")
print(linalg.eigvalsh(gram))

# The Moore-Penrose pseudoinverse of a Gram matrix
gram_pinv = linalg.pinv(gram)

# ── Stratified subsampling of the naturalistic pool ───────────────────────
# Keep ALL of books (and core_clinical); draw redsm5 up to TARGET_PER_SYMPTOM
# per symptom so every symptom lands at the same count (deterministic via SEED).
# Knobs (SEED, TARGET_PER_SYMPTOM) live in the datasets-config cell above.

SEED = 0
_books_n = defaultdict(int)
for _f in os.listdir(datasets["books"]["files"]):
    _books_n[_f.split(",")[1]] += 1
# None -> books' largest class (keeps 100% of books); otherwise the set value.
_target = max(_books_n.values())

def select_files(text_dir, dataset_name):
    """Filenames to include for a dataset: the full corpus for core_clinical and
    books; redsm5 is stratified per symptom up to _target, filling only the
    deficit left by books. Deterministic given SEED and the sorted file list."""
    files = sorted(os.listdir(text_dir))
    if dataset_name != "redsm5":
        return files
    by_symptom = defaultdict(list)
    for f in files:
        by_symptom[f.split(",")[1]].append(f)
    rng = random.Random(SEED) # fixed for reproducibility
    selected = []
    for symptom in sorted(by_symptom):
        need = max(0, _target - _books_n[symptom])
        selected += rng.sample(by_symptom[symptom], min(need, len(by_symptom[symptom])))
    return sorted(selected)

rows = []
for dataset_name, dataset_info in datasets.items():
    folder = dataset_info['folder']
    text_dir = dataset_info['files']

    for text in select_files(text_dir, dataset_name):
        rs = load(folder + text + '_tensor')[0][1:]  # residual stream (the centroid is its mean)
        centroid_rs = centroid(rs.double())  # float64 BEFORE averaging

        raw_similarities = symptom_stack @ centroid_rs  # raw dot products (3,)
        symptom_weights = gram_pinv @ raw_similarities  # decorrelated coefficients (3,)

        with open(os.path.join(text_dir, text), encoding="utf-8") as f:
            plain_text = f.read()

        rows.append([text,
                     symptom_weights[0].item(),  # mood (decorrelated)
                     symptom_weights[1].item(),  # somatic (decorrelated)
                     symptom_weights[2].item(),  # suicidality (decorrelated)
                     text.split(",")[1],
                     dataset_name,
                     plain_text]
                    )

# Naturalistic composition after sampling (books kept in full, redsm5 topped up).
print(f"\nNaturalistic composition (SEED={SEED}, target={_target}/symptom):")
for _sym in ["mood", "somatic", "suicidality"]:
    _b = sum(1 for r in rows if r[5] == "books" and r[4] == _sym)
    _d = sum(1 for r in rows if r[5] == "redsm5" and r[4] == _sym)
    print(f"  {_sym:12s} books={_b:3d}  redsm5={_d:3d}  total={_b + _d:3d}")

tensor([[1.7771e+08, 1.7836e+08, 1.8899e+08],
        [1.7836e+08, 1.7910e+08, 1.8975e+08],
        [1.8899e+08, 1.8975e+08, 2.0144e+08]], dtype=torch.float64)
cond(G) = 14611.6
tensor([3.8185e+04, 2.6693e+05, 5.5794e+08], dtype=torch.float64)

Naturalistic composition (SEED=0, target=141/symptom):
  mood         books=141  redsm5=  0  total=141
  somatic      books= 53  redsm5= 88  total=141
  suicidality  books= 35  redsm5=106  total=141


In [6]:
import textwrap

projection_gram = DataFrame(rows, columns=["filename", "mood", "somatic", "suicidality", "clinician_annotation", "dataset", "plain_text"])

# Visualization-only grouping: fold the held-out corpora (books + redsm5) into a
# single "naturalistic" dataset. Downstream figures/stats use only
# {core_clinical, naturalistic}, so any label left unmapped here is silently
# dropped (cast to NaN by the Categorical) — hence books must be mapped too.
projection_gram["dataset"] = projection_gram["dataset"].replace({"books": "naturalistic", "redsm5": "naturalistic"})


def wrap_for_hover(text, width=80):
    """Insert <br> breaks so long pain_text wraps inside Plotly hover tooltips.
    Plotly hover honours <br> (not raw \n), so wrap each line to `width`
    characters and rejoin with <br>, preserving existing paragraph breaks."""
    if not isinstance(text, str):
        return text
    lines = []
    for line in text.splitlines() or [text]:
        lines.extend(textwrap.wrap(line, width=width) or [""])
    return "<br>".join(lines)


# Pre-wrapped text used only for hover labels (keeps plain_text intact for export)
projection_gram["hover_text"] = projection_gram["plain_text"].map(wrap_for_hover)

# ── Display labels ─────────────────────────────────────────────────────────
# Formal paper text for every figure and table. Data keys stay lowercase
# snake_case; only what a reader sees is title-cased, so filtering/grouping code
# is unaffected. Added as columns so Plotly picks them up for legends directly.
DATASET_LABEL = {"core_clinical": "Core Clinical · In-Sample", "naturalistic": "Naturalistic · Held-Out"}
ANNOTATION_LABEL = {"mood": "Mood", "somatic": "Somatic", "suicidality": "Suicidality"}

projection_gram["Corpus"] = projection_gram["dataset"].map(DATASET_LABEL)
projection_gram["Annotation"] = projection_gram["clinician_annotation"].map(ANNOTATION_LABEL)

In [ ]:
# ── Color palette (consistent across all paper figures) ────────────────────
# Colourblind-safe Okabe-Ito palette (Okabe & Ito 2008): each annotation is
# mapped to its closest Okabe-Ito hue. Vermillion (#D55E00) is the shared
# paper highlight, marking suicidality (the most clinically severe axis), with
# blue/bluish-green for mood/somatic. Chosen over a literal green+red scheme
# because green/red collapses under deutan/protan vision; this set keeps every
# pair distinguishable for red-green colourblind readers.
COLOR_MAP = {
    "Mood": "#0072B2",         # blue
    "Somatic": "#009E73",      # bluish green
    "Suicidality": "#D55E00",  # vermillion (paper highlight)
}
FONT = dict(family="Arial, sans-serif", size=11, color="black")

# Figures are exported straight into the manuscript folder; npj style forbids
# in-figure titles (the caption carries them), so no figure here sets one.
FIGURE_DIR = "manuscript"

# ── Axis styling shared by the three scene axes ────────────────────────────
SCENE_AXIS = dict(
    backgroundcolor="rgba(0,0,0,0)",
    gridcolor="rgba(200,200,200,0.3)",
    zerolinecolor="rgba(150,150,150,0.4)",
    linecolor="black", linewidth=0.75,
    title_font=dict(size=11, family="Arial, sans-serif"),
    tickfont=dict(size=9, family="Arial, sans-serif"),
)

proj_3d = px.scatter_3d(projection_gram,
                        x="mood",
                        y="somatic",
                        z="suicidality",
                        color="Annotation",
                        symbol="Corpus",
                        opacity=0.85,
                        hover_data=["filename", "hover_text"],
                        symbol_map={
                            # keyed off DATASET_LABEL, not literals: the Corpus column holds
                            # those exact strings, so an edit there would silently unmap the symbols.
                            DATASET_LABEL["core_clinical"]: "square",
                            DATASET_LABEL["naturalistic"]: "circle",
                        },
                        color_discrete_map=COLOR_MAP)

proj_3d.update_traces(marker=dict(size=4, line=dict(width=0)))

proj_3d.update_layout(
    template="plotly_white",
    font=FONT,
    legend=dict(
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="rgba(0,0,0,0.2)",
        borderwidth=0.5,
        font=dict(size=10),
        itemsizing="constant",
    ),
    margin=dict(l=0, r=0, t=10, b=10),
    width=720, height=560,
    scene=dict(
        aspectmode="cube",   # equal-length axes regardless of β range (keeps the narrow somatic axis from squishing)
        xaxis=dict(title_text="Mood", **SCENE_AXIS),
        yaxis=dict(title_text="Somatic", **SCENE_AXIS),
        zaxis=dict(title_text="Suicidality", **SCENE_AXIS),
    ),
)

proj_3d.show(renderer="browser")


In [ ]:
# ── Figure 4 (vector export): the same symptom subspace, drawn in matplotlib ──
# The plotly scene above is WebGL, so kaleido rasterizes it and drops z-axis
# tick labels and the axis title on static export; matplotlib redraws the same
# cloud as true vector for the manuscript. The plotly figure stays for
# interactive inspection (hover text); this one is what the paper ships.
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)
from matplotlib.ticker import MaxNLocator

plt.rcParams["font.family"] = ["Arial", "DejaVu Sans"]

MARKER = {"core_clinical": "s", "naturalistic": "o"}

# Native width matched to panels a-d (~5.8 in) so every figure in the paper
# is scaled by roughly the same factor on the page and the type sizes agree.
fig3d = plt.figure(figsize=(5.9, 3.9))
# Bottom margin has to clear the rotated "Somatic" tick labels and axis
# title: this figure is saved without bbox_inches="tight" (that crops the
# z-axis title away), so anything outside the axes rect is lost -- which is
# why the rect is tuned by measuring the exported PDF rather than guessed.
# [0.04, 0.11, 0.68, 0.85] left 0.22/0.17/0.21 in of dead canvas on the
# left/top/bottom; this leaves ~0.07 in a side. Pulling the left edge in to
# 0.005 clips the z tick labels outright, so 0.018 is close to the floor.
ax3d = fig3d.add_axes([0.018, 0.07, 0.702, 0.905], projection="3d")

for annotation in ["mood", "somatic", "suicidality"]:
    for dataset in ["core_clinical", "naturalistic"]:
        sub = projection_gram[(projection_gram["clinician_annotation"] == annotation) &
                              (projection_gram["dataset"] == dataset)]
        ax3d.scatter(sub["mood"], sub["somatic"], sub["suicidality"],
                     marker=MARKER[dataset], s=13, linewidths=0, alpha=0.85,
                     color=COLOR_MAP[ANNOTATION_LABEL[annotation]],
                     label=f"{ANNOTATION_LABEL[annotation]}, {DATASET_LABEL[dataset]}")

ax3d.set_xlabel("Mood", labelpad=10, fontsize=11)
ax3d.set_ylabel("Somatic", labelpad=10, fontsize=11)
ax3d.set_zlabel("Suicidality", labelpad=6, fontsize=11)
ax3d.tick_params(labelsize=8.5, pad=1)
# The two floor axes run diagonally across the page, so their labels sit
# close together and the default tick density collides. The z axis is
# vertical and stays legible at its own spacing.
ax3d.xaxis.set_major_locator(MaxNLocator(5))
ax3d.yaxis.set_major_locator(MaxNLocator(5))
ax3d.view_init(elev=20, azim=45)   # mirrors the plotly camera: mood right, somatic left, suicidality up
ax3d.set_box_aspect((1, 1, 1))     # matches plotly's aspectmode="cube"
for pane_axis in (ax3d.xaxis, ax3d.yaxis, ax3d.zaxis):
    pane_axis.pane.set_facecolor("white")
    pane_axis.pane.set_edgecolor("#bbbbbb")
    pane_axis.pane.set_alpha(1.0)
    pane_axis._axinfo["grid"].update(color="#dddddd", linewidth=0.5)

# Legend in figure coordinates (not axes): a 3-D axes reports only its own box,
# so an axes-anchored legend — and savefig(bbox_inches="tight") — would crop the
# rotated z-axis title away.
fig3d.legend(*ax3d.get_legend_handles_labels(), loc="upper right",
             bbox_to_anchor=(0.995, 0.99), frameon=True, framealpha=0.9,
             edgecolor="#cccccc", fontsize=8.5, markerscale=1.6, handletextpad=0.4,
             borderpad=0.5, labelspacing=0.55, title="Annotation, Corpus", title_fontsize=9.5)

fig3d.savefig(os.path.join(FIGURE_DIR, "4.subspace.pdf"))
plt.show()


In [ ]:
# β-coefficient box plots as a single 2 × 3 grid: columns are the symptom axes
# (a mood, b somatic, c suicidality) side by side, rows are the two corpora.
# The figure sits above the rank matrix (panel d) in the paper. y-scale is
# shared down each column; every facet carries pairwise significance stars.
# No in-figure title: the caption carries it.
import pandas as pd

SYMPTOMS = ["mood", "somatic", "suicidality"]
DATASETS = ["core_clinical", "naturalistic"]
ANNOTS = ["mood", "somatic", "suicidality"]
N_ROWS, N_COLS = len(DATASETS), len(SYMPTOMS)   # grid: corpus down, symptom axis across

# Page geometry (pixels; kaleido writes 0.75 pt per px). Wide enough for three
# axis columns; printed near this size, so the type needs no rescaling.
FIG_W, FIG_H = 680, 425
# Significance-bracket geometry in pixels of the rendered canvas (see
# build_grid_figure): clearance between the data and the first bracket, room per
# stacked level, height of the end ticks, and clearance above the top label.
BRACKET_BASE_PX, BRACKET_GAP_PX, TICK_DROP_PX, LABEL_HEAD_PX = 11, 16, 5, 11

# Melt the three coefficient columns into one facet-row dimension.
long = projection_gram.melt(
    id_vars=["filename", "clinician_annotation", "Annotation", "dataset", "hover_text"],
    value_vars=SYMPTOMS, var_name="symptom_axis", value_name="coefficient",
)
long["symptom_axis"] = pd.Categorical(long["symptom_axis"], categories=SYMPTOMS, ordered=True)
long["dataset"] = pd.Categorical(long["dataset"], categories=DATASETS, ordered=True)

# Fold per-group N into the x tick labels (counts from projection_gram, not the melt).
_n = projection_gram.groupby(["dataset", "clinician_annotation"], observed=True).size()

def _annot_label(dataset, annotation):
    return f"{ANNOTATION_LABEL[annotation]}<br>(n={_n[(dataset, annotation)]})"

long["annot_label"] = [_annot_label(d, a)
                       for d, a in zip(long["dataset"], long["clinician_annotation"])]
# Labels present in each column, pinned per-axis below (a global category_orders
# would stamp phantom ticks for the other columns' labels and squeeze the boxes).
_col_labels = {d: [_annot_label(d, a) for a in ANNOTS if (d, a) in _n] for d in DATASETS}

# ── Pairwise significance, both columns ────────────────────────────────────
# Dunn's post-hoc on the three annotation-group pairs per axis, computed
# separately within each dataset (core_clinical and naturalistic each form
# their own family of 3 pairs × 3 axes = 9 tests, BH-corrected within that
# family — the two datasets answer different questions, so their corrections
# aren't pooled). Brackets show stars only (*** <.001, ** <.01, * <.05, ns
# otherwise); exact p_adj and Cliff's δ print to stdout for the caption.
import numpy as np
from scipy.stats import norm, rankdata, mannwhitneyu

_ANNOS = ["mood", "somatic", "suicidality"]   # x positions 0, 1, 2
_PAIRS = [(0, 1), (1, 2), (0, 2)]              # stacked low -> high

def _dunn_p(groups):
    """Tie-corrected Dunn z-test: two-sided p for each pair (i<j)."""
    allv = np.concatenate(groups); N = allv.size
    ranks = rankdata(allv)
    edges = np.cumsum([0] + [g.size for g in groups])
    rbar = [ranks[edges[k]:edges[k + 1]].mean() for k in range(len(groups))]
    n = [g.size for g in groups]
    _, cnt = np.unique(allv, return_counts=True)
    sigma2 = (N * (N + 1) / 12.0) - np.sum(cnt ** 3 - cnt) / (12.0 * (N - 1))
    return {(i, j): 2 * norm.sf(abs((rbar[i] - rbar[j]) / np.sqrt(sigma2 * (1 / n[i] + 1 / n[j]))))
            for i in range(len(groups)) for j in range(i + 1, len(groups))}

def _cliffs_delta(a, b):
    """Cliff's δ via Mann-Whitney U: +1 = a wholly above b, 0 = full overlap."""
    return 2 * mannwhitneyu(a, b, alternative="two-sided").statistic / (a.size * b.size) - 1

def _bh(pvals):
    """Benjamini-Hochberg FDR-adjusted p-values."""
    p = np.asarray(pvals, float); m = p.size; o = np.argsort(p); r = p[o]
    adj = np.minimum.accumulate((r * m / np.arange(1, m + 1))[::-1])[::-1]
    out = np.empty(m); out[o] = np.clip(adj, 0, 1)
    return out

def _stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

def _dataset_stats(dataset_name):
    """Per-dataset Dunn's pairwise + BH correction across its own 9 tests."""
    sub = projection_gram[projection_gram["dataset"] == dataset_name]
    samples = {ax: [sub.loc[sub["clinician_annotation"] == an, ax].to_numpy() for an in _ANNOS]
               for ax in SYMPTOMS}
    raw, delta = {}, {}
    for ax in SYMPTOMS:
        dp = _dunn_p(samples[ax])
        for pr in _PAIRS:
            raw[(ax, pr)] = dp[pr]
            delta[(ax, pr)] = _cliffs_delta(samples[ax][pr[0]], samples[ax][pr[1]])
    keys = [(ax, pr) for ax in SYMPTOMS for pr in _PAIRS]
    adj = dict(zip(keys, _bh([raw[k] for k in keys])))
    return sub, adj, delta

_stats_by_dataset = {d: _dataset_stats(d) for d in DATASETS}

for _d in DATASETS:
    _, _adj, _delta = _stats_by_dataset[_d]
    print(f"Dunn's pairwise ({_d}), BH-adjusted across 9 tests:")
    for _ax in SYMPTOMS:
        for _pr in _PAIRS:
            _a, _b = _ANNOS[_pr[0]], _ANNOS[_pr[1]]
            print(f"  beta_{_ax:11s} {_a:>11s} vs {_b:<11s}  p_adj={_adj[(_ax, _pr)]:.2e}"
                  f"  {_stars(_adj[(_ax, _pr)]):>3s}  delta={_delta[(_ax, _pr)]:+.2f}")


def build_grid_figure():
    """Panels a-c side by side: columns are the three symptom axes, rows the two
    corpora, so all six box plots read as one figure. The y-scale is shared down
    each column (the axes have different ranges, the corpora do not), and every
    facet carries its own significance brackets."""
    fig = px.box(long, y="coefficient", x="annot_label", color="Annotation",
                 facet_col="symptom_axis", facet_row="dataset",
                 facet_col_spacing=0.055, facet_row_spacing=0.155,   # room for the lower row's bracket stack under the upper row's tick labels
                 points="all", hover_data=["filename", "hover_text"],
                 category_orders={"symptom_axis": SYMPTOMS, "dataset": DATASETS},
                 color_discrete_map=COLOR_MAP)

    # Facet axis numbering is row-major from the BOTTOM row: idx = r*N_COLS+c+1,
    # so r=0 is the bottom row (naturalistic) and r=N_ROWS-1 the top (core clinical).
    def _ax(r, c, kind):
        idx = r * N_COLS + c + 1
        return f"{kind}axis" if idx == 1 else f"{kind}axis{idx}"

    def _row_of(dataset):
        """Grid row index (0 = bottom) for a dataset; DATASETS reads top-down."""
        return N_ROWS - 1 - DATASETS.index(dataset)

    # Independent x per facet so each row shows its own n= tick labels.
    fig.update_xaxes(matches=None)
    fig.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels=True))
    for d in DATASETS:
        for c in range(N_COLS):
            fig.layout[_ax(_row_of(d), c, "x")].update(categoryorder="array",
                                                       categoryarray=_col_labels[d])

    # px links every facet's y to the first, which would force all three axes
    # onto one scale. Drop the links entirely and give each column an explicit
    # shared range instead (computed with the brackets, below): linked axes
    # ignore their own range setting, and the tick labels then disagree with the
    # data that kaleido draws.
    fig.update_yaxes(matches=None)
    # px also hides the tick labels of every column but the first while the axes
    # are linked; unlinking them without this leaves B and C with bare ticks.
    fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))

    fig.update_layout(
        template="plotly_white",
        font={**FONT, "size": 11},
        showlegend=False,   # x tick labels already name every annotation
        boxmode="overlay",   # x==color keeps full-width boxes (group mode would split them)
        boxgap=0.4, boxgroupgap=0.3,
        margin=dict(l=58, r=34, t=32, b=50),   # r: just enough for the rotated corpus labels
        # Kept in sync with the write_image() call below: kaleido IGNORES
        # layout.width/height and renders at its own default unless the size is
        # passed to write_image, but the margins above are absolute pixels, so
        # the layout still has to be told the real canvas size.
        width=FIG_W, height=FIG_H,
    )

    fig.update_traces(
        offsetgroup=None, alignmentgroup=None,   # only matter in group mode
        marker=dict(size=2.5, opacity=0.5, line=dict(width=0)),
        line=dict(width=1.4), jitter=0.3, pointpos=0,
    )

    # Column titles become "<b>A</b> Mood"; row titles keep the corpus label on
    # the right edge (px writes them rotated there).
    _letter_of = dict(zip(SYMPTOMS, ["A", "B", "C"]))

    def _tidy_facet(a):
        label = a.text.split("=")[-1]
        if label in SYMPTOMS:
            a.update(text=f"<b>{_letter_of[label]}</b>  {ANNOTATION_LABEL[label]} Axis",
                     font=dict(size=13, color="black", family="Arial, sans-serif"))
        else:
            a.update(text=DATASET_LABEL.get(label, label),
                     font=dict(size=11, color="black", family="Arial, sans-serif"))

    fig.for_each_annotation(_tidy_facet)

    fig.update_xaxes(title_text="", tickfont=dict(size=10), tickangle=0, showgrid=False,
                     linecolor="black", linewidth=0.6, ticks="outside", ticklen=3,
                     tickwidth=0.6, automargin=True)
    fig.update_yaxes(title_text="", title_font=dict(size=11), tickfont=dict(size=10),
                     title_standoff=4, automargin=True, showgrid=True,
                     gridcolor="rgba(200,200,200,0.3)", gridwidth=0.5, zeroline=True,
                     zerolinecolor="rgba(150,150,150,0.4)", zerolinewidth=0.5,
                     linecolor="black", linewidth=0.6, ticks="outside", ticklen=3,
                     tickwidth=0.6)
    # y-title on every row, not just the bottom one: each row is its own corpus
    # on its own axis, so both need the units named at their left.
    for c in range(N_COLS):
        for r in range(N_ROWS):
            fig.layout[_ax(r, c, "y")].update(title_text="Coefficient (β)",
                                              title_font=dict(size=11))

    # Three stacked brackets per facet (stars only), then one explicit range per
    # column applied to BOTH its rows — that is what makes the corpora
    # comparable down a column without relying on axis links.
    # Bracket geometry is solved in PIXELS, not in fractions of the data span:
    # three stacked brackets in a short facet crowd their own labels otherwise,
    # and a span-relative base offset lets the end ticks dip into the topmost
    # points. With a facet plotting height H, a base offset B above the data, a
    # per-level gap G and clearance L above the top label, the axis range that
    # reserves exactly that much room is
    #     R = 1.05*span / (1 - (B + 2G + L)/H)
    # after which one pixel is worth R/H data units.
    _facet_h_px = (FIG_H - 32 - 50) * (1 - 0.155) / N_ROWS

    for _c, _axis in enumerate(SYMPTOMS):
        _col = projection_gram[_axis]                   # both corpora, one axis
        _lo, _hi = float(_col.min()), float(_col.max())
        _span = _hi - _lo
        _reserved = (BRACKET_BASE_PX + 2 * BRACKET_GAP_PX + LABEL_HEAD_PX) / _facet_h_px
        _unit = (1.05 * _span / (1 - _reserved)) / _facet_h_px   # data units per pixel
        _base = BRACKET_BASE_PX * _unit                 # clears the end ticks off the data
        _step = BRACKET_GAP_PX * _unit
        _drop = TICK_DROP_PX * _unit
        _needed_hi = _hi

        for _d in DATASETS:
            _sub, _adj, _delta = _stats_by_dataset[_d]
            _idx = _row_of(_d) * N_COLS + _c + 1
            _xref, _yref = f"x{_idx}", f"y{_idx}"
            _top = float(_sub[_axis].max())             # this corpus's own ceiling
            for _lvl, (_x0, _x1) in enumerate(_PAIRS):
                _y = _top + _base + _lvl * _step
                fig.add_shape(type="line", xref=_xref, yref=_yref, x0=_x0, x1=_x1, y0=_y, y1=_y,
                              line=dict(color="black", width=0.9))
                for _xx in (_x0, _x1):
                    fig.add_shape(type="line", xref=_xref, yref=_yref, x0=_xx, x1=_xx,
                                  y0=_y, y1=_y - _drop, line=dict(color="black", width=0.9))
                _txt = _stars(_adj[(_axis, (_x0, _x1))])
                # Anchored to the BOTTOM of its own bracket and nudged in pixels:
                # asterisk ink sits high in the em box, so a middle anchor drifts
                # up towards the next bracket while "ns" sinks onto its own line.
                fig.add_annotation(xref=_xref, yref=_yref, x=(_x0 + _x1) / 2, y=_y,
                                   yshift=(2 if _txt == "ns" else -3),
                                   text=_txt, showarrow=False, xanchor="center", yanchor="bottom",
                                   font=dict(size=(11 if _txt != "ns" else 8),
                                             family="Arial, sans-serif", color="black"))
            _needed_hi = max(_needed_hi,
                             _top + _base + 2 * _step + LABEL_HEAD_PX * _unit)

        _range = [_lo - 0.05 * _span, _needed_hi]
        for _r in range(N_ROWS):
            fig.layout[_ax(_r, _c, "y")].range = _range

    return fig


# Panels a-c as one wide figure. write_image MUST carry width/height: kaleido
# ignores layout.width/height and would otherwise emit its 700x500 default.
fig_abc = build_grid_figure()
fig_abc.write_image(os.path.join(FIGURE_DIR, "3.coefficient_abc.pdf"),
                    width=FIG_W, height=FIG_H)

fig_abc.show(renderer="browser")


In [9]:
# ── Table 2: Per-condition β coefficient summary ─────────────────────────
# Companion to the three box plots above. Reports median (Q1, Q3) and N for
# each (dataset, clinician_annotation) combination across all three β axes,
# so readers can read exact values without cluttering the figures.

import pandas as pd

# Condition order matches reading flow: anchors → naturalistic
condition_order = [
    ("core_clinical", "mood"),
    ("core_clinical", "somatic"),
    ("core_clinical", "suicidality"),
    ("naturalistic", "mood"),
    ("naturalistic", "somatic"),
    ("naturalistic", "suicidality"),
]

def fmt_iqr(values):
    """Format a numeric series as 'median (Q1, Q3)' with signed two-decimal precision."""
    med = values.median()
    q1, q3 = values.quantile([0.25, 0.75])
    return f"{med:+.2f} ({q1:+.2f}, {q3:+.2f})"

rows_data = []
index_tuples = []
for dataset, annotation in condition_order:
    subset = projection_gram[
        (projection_gram["dataset"] == dataset) &
        (projection_gram["clinician_annotation"] == annotation)
    ]
    index_tuples.append((DATASET_LABEL[dataset], ANNOTATION_LABEL[annotation]))
    rows_data.append([
        len(subset),
        fmt_iqr(subset["mood"]),
        fmt_iqr(subset["somatic"]),
        fmt_iqr(subset["suicidality"]),
    ])

index = pd.MultiIndex.from_tuples(index_tuples, names=["Dataset", "Annotation"])
columns = pd.MultiIndex.from_tuples([
    ("",              "N"),
    ("β Mood",        "median (Q1, Q3)"),
    ("β Somatic",     "median (Q1, Q3)"),
    ("β Suicidality", "median (Q1, Q3)"),
])

table_2 = pd.DataFrame(rows_data, index=index, columns=columns)
table_2.to_csv("Table_2_projection_beta_summary.csv")

# Save styled HTML for paper/supplementary use (open in any browser)
styled_table_2 = (
    table_2.style
    .set_table_styles([
        {'selector': 'caption',
         'props': [('caption-side', 'top'),
                   ('font-family', 'Arial, sans-serif'),
                   ('font-size', '10pt'),
                   ('font-weight', 'normal'),
                   ('text-align', 'left'),
                   ('padding', '6px 0 12px 0'),
                   ('color', '#333'),
                   ('line-height', '1.4')]},
        {'selector': 'table',
         'props': [('font-family', 'Arial, sans-serif'),
                   ('font-size', '10pt'),
                   ('border-collapse', 'collapse')]},
        {'selector': 'th',
         'props': [('font-family', 'Arial, sans-serif'),
                   ('font-weight', '600'),
                   ('text-align', 'center'),
                   ('padding', '6px 12px'),
                   ('border-bottom', '0.75px solid #555'),
                   ('background-color', '#fafafa')]},
        {'selector': 'thead tr:first-child th',
         'props': [('font-weight', '700'),
                   ('font-size', '10pt'),
                   ('background-color', '#f3f3f3'),
                   ('border-bottom', '0.5px solid #aaa')]},
        {'selector': 'td',
         'props': [('text-align', 'center'),
                   ('padding', '6px 12px'),
                   ('border-bottom', '0.5px solid #eee')]},
        {'selector': 'th.row_heading',
         'props': [('text-align', 'left'),
                   ('background-color', '#fafafa'),
                   ('font-weight', '500')]},
    ])
    .set_caption(
        "Table 2. Per-condition decorrelated symptom projection coefficients, "
        "grouped by dataset. Median (Q1, Q3) reported for each β axis across "
        "corpus × clinician annotation."
    )
)
with open("Table_2_projection_beta_summary.html", "w", encoding="utf-8") as f:
    f.write(styled_table_2.to_html())

table_2


β Mood  \
                                         N       median (Q1, Q3)   
Dataset                   Annotation                               
Core Clinical · In-Sample Mood          19  +0.77 (+0.49, +1.14)   
                          Somatic       24  -0.19 (-0.53, +0.21)   
                          Suicidality    8  -0.05 (-0.27, +0.09)   
Naturalistic · Held-Out   Mood         141  -0.08 (-0.33, +0.15)   
                          Somatic      141  -0.64 (-1.04, -0.33)   
                          Suicidality  141  -0.74 (-1.30, -0.28)   

                                                  β Somatic  \
                                            median (Q1, Q3)   
Dataset                   Annotation                          
Core Clinical · In-Sample Mood         +0.04 (-0.06, +0.20)   
                          Somatic      +1.01 (+0.86, +1.27)   
                          Suicidality  +0.04 (-0.06, +0.08)   
Naturalistic · Held-Out   Mood         +0.22 (+0.11, +0.33)   
                          Somatic      +0.74 (+0.61, +0.88)   
                          Suicidality  +0.34 (+0.23, +0.46)   

                                              β Suicidality  
                                            median (Q1, Q3)  
Dataset                   Annotation                         
Core Clinical · In-Sample Mood         +0.10 (-0.00, +0.30)  
                          Somatic      +0.14 (-0.14, +0.38)  
                          Suicidality  +0.96 (+0.86, +1.34)  
Naturalistic · Held-Out   Mood         +0.85 (+0.66, +1.08)  
                          Somatic      +0.91 (+0.58, +1.28)  
                          Suicidality  +1.40 (+0.95, +1.89)

In [ ]:
# ── Figure: symptom projection confusion matrices (rank-based) ────────────────
# Rows = clinician annotation, columns = projection axis. Within each column the
# three annotation groups are RANKED by median β (rank 1 = highest), so cell
# colour shows whether the projection's strongest group on an axis matches the
# clinician label. A clean rank-1 diagonal = agreement. Core clinical is the
# in-sample check; naturalistic is the held-out generalization test. Recomputed
# from `projection_gram`; printed numbers are the underlying median β, with
# per-group N beside each row.

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams["font.family"] = ["Arial", "DejaVu Sans"]

ANNOS = ["mood", "somatic", "suicidality"]
AXES  = ["mood", "somatic", "suicidality"]
PANELS = [("Core Clinical", "In-sample", "core_clinical"),
          ("Naturalistic",  "Held-out",  "naturalistic")]
# Sequential green ramp anchored on Okabe-Ito bluish green (#009E73), matching the paper
# palette. Rank encoded by luminance (dark = rank 1 / highest), so the scale is
# colourblind-safe; (fill, text) pairs keep label contrast above WCAG AA.
RANK_STYLE = {1: ("#009E73", "white"), 2: ("#80CEB9", "#06402F"), 3: ("#D1EEE6", "#06402F")}
FRAME_LW = 0.7            # block outline; a touch heavier than the internal grid
# Hairline black rule between cells. Stroking every cell also removes the 1 px
# white seam that butt-jointed vector rectangles leave when they merely share an
# edge (verified at 600 dpi). Nature/npj set 0.25 pt as the minimum line weight
# AT FINAL PRINTED SIZE, and this canvas is scaled by PAGE_SCALE on the page, so
# the nominal weight has to clear 0.25 / PAGE_SCALE = 0.29 pt: 0.35 pt nominal
# lands at 0.30 pt printed, fine but never hairline-dropout.
GRID, GRID_LW = "#000000", 0.35

def median_and_rank(dataset):
    med = {an: {ax: projection_gram[(projection_gram["dataset"] == dataset) &
                                    (projection_gram["clinician_annotation"] == an)][ax].median()
                for ax in AXES} for an in ANNOS}
    rank = {an: {} for an in ANNOS}
    for ax in AXES:
        col = pd.Series({an: med[an][ax] for an in ANNOS})
        r = col.rank(ascending=False, method="min").astype(int)
        for an in ANNOS:
            rank[an][ax] = int(r[an])
    return med, rank

# Type sizes follow the Nature Portfolio figure guide: every label between 5 and
# 7 pt AT FINAL PRINTED SIZE; the panel letter sits above that window at 9 pt so
# it matches A/B/C on the a-c grid rather than the guide. This canvas is 7.67 in of
# ink wide and is placed at width=\textwidth (469.8 pt = 6.52 in), so it is
# scaled by 0.851 on the page -- the nominal sizes below are therefore the
# on-page targets divided by that factor. Keep the two in sync if the crop or
# \textwidth changes: PAGE_SCALE is printed after savefig as a check.
PAGE_SCALE = 0.851                             # \textwidth / natural canvas width
_pt = lambda on_page: round(on_page / PAGE_SCALE, 1)
FS_TITLE  = _pt(7.0)                           # corpus name
FS_SUB    = _pt(6.0)                           # in-sample / held-out
FS_HEAD   = _pt(6.3)                           # β-axis column headers
FS_ROW    = _pt(7.0)                           # annotation row labels
FS_VALUE  = _pt(7.0)                           # median β inside each cell
FS_SMALL  = _pt(5.5)                           # rank marks, n = ...
FS_LEGEND = _pt(6.0)                           # rank key
FS_LETTER = _pt(9.0)                           # panel letter; matches A/B/C on the a-c grid,
                                               # which are 13 pt on a 680 pt canvas -> 9.0 pt on page

CW, CH = 1.05, 0.95
LABEL_W, PANEL_GAP, HEAD_DY, TITLE_DY = 1.55, 1.05, 0.30, 0.95
panel_x0 = [LABEL_W, LABEL_W + len(AXES) * CW + PANEL_GAP]
yc = [0.0, -CH, -2 * CH]                       # row centres (mood, somatic, suicidality)

def draw_cell(ax, x, y, med, rank, diagonal):
    # Butt-jointed cells divided by a hairline black rule, as in a conventional
    # heat map: no rounded corners, no gutters, no shadow. The rule keeps two
    # same-rank neighbours from reading as one block of colour.
    #
    # The rank is hashed (#1) rather than bare: directly under a signed β, a lone
    # numeral would read as part of the number. The key spells the same ranks out
    # as ordinals instead, where there is nothing to be confused with.
    fill, tc = RANK_STYLE[rank]
    ax.add_patch(Rectangle((x, y - CH / 2), CW, CH, linewidth=GRID_LW,
                           edgecolor=GRID, facecolor=fill, zorder=2))
    ax.text(x + CW / 2, y + 0.10, f"{med:+.2f}", ha="center", va="center",
            fontsize=FS_VALUE, color=tc, zorder=4)
    ax.text(x + CW / 2, y - 0.20, f"#{rank}", ha="center", va="center",
            fontsize=FS_SMALL, color=tc, zorder=4)
    if diagonal:                                # matching annotation/axis pair
        ax.add_patch(Rectangle((x + 0.02, y - CH / 2 + 0.02), CW - 0.04, CH - 0.04,
                               linewidth=1.6, edgecolor="#1a1a1a", facecolor="none",
                               zorder=5))

# Native width matched to the a-c grid (680 px = 7.08 in) so both halves of the
# figure print at the same fraction of \textwidth: equal type scale AND equal
# ink width, instead of D sitting narrower with white margins either side.
fig, ax = plt.subplots(figsize=(7.1, 4.4))

for r, an in enumerate(ANNOS):                 # shared annotation row labels
    ax.text(LABEL_W - 0.20, yc[r], ANNOTATION_LABEL[an], ha="right", va="center", fontsize=FS_ROW)

# Row-axis title. A horizontal "Annotation" sitting above the row labels reads as
# a column header for whatever is beneath it, which is exactly the confusion it
# has to prevent -- rotated and centred on the three rows, it unambiguously names
# the vertical dimension instead. Both panels share it, as they share the labels.
# Kept right of the panel letter so D stays the leftmost ink and therefore stays
# aligned with A on the grid above (see the crop note below).
ax.text(0.30, yc[1], "Clinician annotation", ha="center", va="center", rotation=90,
        fontsize=FS_HEAD, color="#333333")

for p, (title, sub, dataset) in enumerate(PANELS):
    med, rank = median_and_rank(dataset)
    x0, x1 = panel_x0[p], panel_x0[p] + len(AXES) * CW
    top = yc[0] + CH / 2
    ax.text(x0, top + TITLE_DY, title, ha="left", va="center", fontsize=FS_TITLE)
    ax.text(x0, top + TITLE_DY - 0.30, sub, ha="left", va="center", fontsize=FS_SUB, color="#444444")
    for c, axis in enumerate(AXES):
        ax.text(x0 + c * CW + CW / 2, top + HEAD_DY, f"β {ANNOTATION_LABEL[axis]}",
                ha="center", va="center", fontsize=FS_HEAD, color="#333333")
    for r, an in enumerate(ANNOS):
        for c, axis in enumerate(AXES):
            draw_cell(ax, x0 + c * CW, yc[r], med[an][axis], rank[an][axis], diagonal=(an == axis))
        n = len(projection_gram[(projection_gram["dataset"] == dataset) &
                                (projection_gram["clinician_annotation"] == an)])
        ax.text(x1 + 0.16, yc[r], f"n = {n}", ha="left", va="center",
                fontsize=FS_SMALL, color="#444444")
    # frame enclosing the 3x3 block
    ax.add_patch(Rectangle((x0, yc[-1] - CH / 2), len(AXES) * CW, len(ANNOS) * CH,
                           linewidth=FRAME_LW, edgecolor=GRID, facecolor="none", zorder=6))

# Rank key: one contiguous, framed strip under the left-hand matrix, built from
# the same cell geometry rather than floating rounded swatches. Laid out as a
# single line -- key first, then its label -- the way a colour bar carries its
# caption alongside. Swatches spell the ranks as ordinals; the cells hash them.
ORDINAL = {1: "1st", 2: "2nd", 3: "3rd"}
KW, KH = 0.62, 0.26
# Hung a fixed clearance below the matrices rather than at an absolute y, so the
# gap stays right if the key gains or loses a line.
cy = (yc[-1] - CH / 2) - 0.45 - KH / 2
lx = panel_x0[0]
for i, rk in enumerate([1, 2, 3]):
    fill, tc = RANK_STYLE[rk]
    x = lx + i * KW
    ax.add_patch(Rectangle((x, cy - KH / 2), KW, KH, facecolor=fill,
                           edgecolor=GRID, linewidth=GRID_LW, zorder=2))
    ax.text(x + KW / 2, cy, ORDINAL[rk], ha="center", va="center",
            fontsize=FS_SMALL, color=tc, zorder=4)
ax.add_patch(Rectangle((lx, cy - KH / 2), 3 * KW, KH, facecolor="none",
                       edgecolor=GRID, linewidth=FRAME_LW, zorder=6))
ax.text(lx + 3 * KW + 0.16, cy, "Shading: rank within column; 1st = highest median β",
        ha="left", va="center", fontsize=FS_LEGEND, color="#444444", style="italic")

# Panel letter -- UPPERCASE, matching A/B/C on the box-plot grid above. npj
# journals set panel letters and in-chart annotations in upper case, unlike the
# lower-case Nature house style, so keep this a capital D.
ax.text(-0.15, yc[0] + CH / 2 + TITLE_DY - 0.2, "D", ha="left", va="top",
        fontsize=FS_LETTER, fontweight="bold")

ax.set_xlim(-0.15, panel_x0[1] + len(AXES) * CW + 0.85)
ax.set_ylim(cy - 0.42, yc[0] + CH / 2 + TITLE_DY + 0.18)
ax.set_aspect("equal"); ax.axis("off")
fig.tight_layout(pad=0.5)

# A plain bbox_inches="tight" crops to an equal 0.1 in pad on every side, which
# is why D used to sit ~0.24 in further left than the a-c grid above it: that
# grid is a plotly canvas with fixed l=58/r=34 px margins, so its ink stops well
# inside its own edges. Both halves are placed at width=\textwidth, so they line
# up only if their ink margins are the same FRACTION of the page width. Build
# the crop box explicitly instead.
#
# The ink box has to come from the artists, not fig.get_tightbbox(): with
# axis("off") the latter still returns the axes rectangle, whose right edge sits
# ~0.3 in past the last "n = ..." label, and the right margin would come out too
# wide.
from matplotlib.transforms import Bbox

ABC_FRAC_L, ABC_FRAC_R = 0.0490, 0.0611   # ink margins of 3.coefficient_abc.pdf, as a fraction of its width
PAD_T, PAD_B = 0.10, 0.12                 # in; matches the A-C grid's trimmed margins
TEXTWIDTH_IN = 469.755 / 72.27            # \textwidth of the manuscript, in inches

fig.canvas.draw()
_rend = fig.canvas.get_renderer()
_ink = Bbox.union([a.get_window_extent(_rend) for a in (*ax.texts, *ax.patches, *ax.lines)]
                  ).transformed(fig.dpi_scale_trans.inverted())
_total_w = (_ink.x1 - _ink.x0) / (1 - ABC_FRAC_L - ABC_FRAC_R)

fig.savefig(os.path.join(FIGURE_DIR, "3.coefficient_d.pdf"),
            bbox_inches=Bbox.from_extents(_ink.x0 - ABC_FRAC_L * _total_w, _ink.y0 - PAD_B,
                                          _ink.x1 + ABC_FRAC_R * _total_w, _ink.y1 + PAD_T),
            pad_inches=0)

# Check the assumed page scale against what was actually written, then report the
# on-page size of the smallest and largest label so the 5-7 pt window is verifiable.
_actual = TEXTWIDTH_IN / _total_w
print(f"PAGE_SCALE assumed {PAGE_SCALE:.3f}, actual {_actual:.3f}")
print(f"on-page type: {min(FS_SMALL, FS_SUB) * _actual:.1f}-{FS_TITLE * _actual:.1f} pt"
      f" (letter {FS_LETTER * _actual:.1f} pt)")
plt.show()
